In [ ]:
!nvidia-smi
!nvcc --version | tail -1

import time
import numpy as np
import cupy as cp

print(cp.cuda.runtime.getDeviceProperties(0)["name"].decode(), "| CuPy", cp.__version__)

Wed Oct  7 02:15:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
def medir(fn, reps=5, sync=None):
    fn()
    if sync: sync()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        if sync: sync()
        ts.append(time.perf_counter() - t0)
    return np.mean(ts), np.std(ts)

sync = cp.cuda.Device().synchronize

for n in [256, 1024, 4096]:
    A = np.random.rand(n, n).astype(np.float32)
    B = np.random.rand(n, n).astype(np.float32)
    Ag, Bg = cp.asarray(A), cp.asarray(B)

    t_cpu, s_cpu = medir(lambda: A @ B)
    t_gpu, s_gpu = medir(lambda: Ag @ Bg, sync=sync)
    t_mal, _ = medir(lambda: Ag @ Bg)

    print(f"n={n:5d} | CPU {t_cpu:.4f}s | GPU {t_gpu:.5f}s (speedup {t_cpu/t_gpu:7.1f}x) | sin sync {t_mal:.6f}s")

n=  256 | CPU 0.0064s | GPU 0.00013s (speedup    49.7x) | sin sync 0.000055s
n= 1024 | CPU 0.0362s | GPU 0.00142s (speedup    25.5x) | sin sync 0.000067s
n= 4096 | CPU 1.2490s | GPU 0.04277s (speedup    29.2x) | sin sync 0.000035s
